Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# 3. Change the source table while the streams run

Run this notebook while notebooks 1 and 2 are running. Each section makes one kind of change to the Iceberg source table and shows its effect on:

- **a probe**: a short-lived stream over the source table, with the read options under test. It starts just before the change, using `stream-from-timestamp`, runs for about half a minute, and reports what it delivered or why it failed;
- **the consumer** of notebook 2, which skips `overwrite` and `delete` snapshots;
- **ADW**: `ORDER_EVENTS_ICEBERG`, the external table that reads the source in place, and `ORDERS`, the table the consumer maintains.

Every "Observed" note comes from a run on AIDP Workbench with Spark 3.5.0 and Iceberg 1.8.x (OCI build). The changes target **lab orders** that the producer never writes -- `order_id` 900001 and up, regions starting with `lab` -- so the results do not depend on timing.

Changes that would break notebooks 1 or 2, such as renaming a column or rebuilding the table, run against a small **scratch copy** instead.

The producer commits every minute, and on AIDP's native Iceberg tables two writers that commit at the same moment race for the next metadata version: the loser fails with `CommitFailedException: Version N already exists` once Iceberg's own retries run out. So every write to the live table below goes through `write_sql` or `committed`, which retry on that error only. See also [Concurrent Writes to Native Iceberg Tables](../transformation/iceberg/Iceberg_Concurrent_Writes.ipynb).

## Configuration and helpers

In [ ]:
# ---- shared configuration: keep these values identical in all three notebooks ---------------
CATALOG = "default"
SCHEMA = "iceberg_streaming_demo"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.order_events"

# The Iceberg table lives in your bucket as <WAREHOUSE>/<schema>/<table>. ADW's Hadoop catalog
# support finds a table by that layout (notebook 2 creates the ADW external table).
WAREHOUSE = "oci://<bucket>@<namespace>/iceberg-streaming/warehouse"
TABLE_LOCATION = f"{WAREHOUSE}/{SCHEMA}/order_events"

# Checkpoints and stop files live on a Volume (here an external volume over a bucket folder).
STATE_DIR = "/Volumes/default/default/iceberg_checkpoints/streaming-from-iceberg-source"
CONTROL_DIR = f"{STATE_DIR}/control"

# Workspace folder holding this sample's helper modules, stream_control.py and adw_sink.py.
SAMPLE_DIR = "/Workspace/streaming-from-iceberg-source"

# ---- ADW connection (see the README, "One-time setup") ------------------------------------------
CREDENTIAL_NAME = "adw_iceberg_stream"         # Credential Store entry: keys user, password, wallet_password, tns
WALLET_DIR = "/Volumes/default/default/adw_wallets/<wallet-folder>"   # the unzipped ADW wallet
ADW_OS_CREDENTIAL = "OCI_ICEBERG_CRED"         # DBMS_CLOUD credential in ADW that can read the bucket
OCI_REGION = "us-ashburn-1"                    # region of the bucket
EXTERNAL_TABLE = "ORDER_EVENTS_ICEBERG"        # ADW external table over the Iceberg source table
CONSUMER_QUERY = "orders_to_adw"               # streaming query name, also the key in STREAM_BATCH_LOG
CONSUMER_CHECKPOINT = f"{STATE_DIR}/consumer"  # checkpoint of notebook 2's query

assert "<" not in WAREHOUSE, "Set WAREHOUSE to your bucket and namespace."
import os
assert os.path.isdir(os.path.dirname(STATE_DIR)), f"Create the Volume first (README): {os.path.dirname(STATE_DIR)}"

In [ ]:
import random
import re
import sys
import time
import uuid
from decimal import Decimal

from pyspark.errors import StreamingQueryException
from pyspark.sql import Window, functions as F

if SAMPLE_DIR not in sys.path:
    sys.path.insert(0, SAMPLE_DIR)
import adw_sink
import scratch_tables
import stream_control

adw = adw_sink.AdwConnection.from_credential_store(aidputils.secrets, CREDENTIAL_NAME, WALLET_DIR)
sink = adw_sink.AdwSink(spark, adw, query_name="table_changes")   # same ADW tables as notebook 2
SYSTEM = f"{CATALOG}.system"                  # Iceberg procedures: CALL default.system.<procedure>(...)
PROBE_DIR = f"{STATE_DIR}/probes"             # throwaway checkpoints of the probes
LAB_EVENT_ID_BASE = 10**15                    # event ids of lab events; the producer's ids stay far below


def now_ms():
    return int(time.time() * 1000)


_COMMIT_CONFLICT = re.compile(r"CommitFailedException|Version \d+ already exists")


def committed(write, attempts=6):
    """Run a write to the live table, retrying when it loses a commit race with the producer.

    A failed Iceberg commit changes nothing, so the write can simply run again. Other errors,
    including CommitStateUnknownException, are raised at once.
    """
    for attempt in range(1, attempts + 1):
        try:
            return write()
        except Exception as e:
            if attempt == attempts or not _COMMIT_CONFLICT.search(str(e)):
                raise
            wait = random.uniform(1, 3) * attempt
            print(f"   commit conflict with a concurrent writer; retrying in {wait:.0f}s")
            time.sleep(wait)


def write_sql(statement):
    """spark.sql for statements that change the live source table."""
    return committed(lambda: spark.sql(statement))


def first_line(error):
    """The useful part of a StreamingQueryException message."""
    text = str(error)
    marker = "terminated with exception: "
    text = text.split(marker, 1)[1] if marker in text else text
    return text.strip().splitlines()[0][:220]


def current_snapshot_id(table=SOURCE_TABLE):
    return spark.sql(f"SELECT snapshot_id FROM {table}.refs WHERE name = 'main'").first()[0]


def changes_since(ms):
    """Snapshots committed since `ms` other than the producer's (which are tagged with `app`)."""
    return spark.sql(f"""
        SELECT snapshot_id, operation, committed_at,
               summary['added-records'] AS added_rows, summary['deleted-records'] AS deleted_rows
        FROM {SOURCE_TABLE}.snapshots
        WHERE committed_at >= timestamp_millis({ms}) AND summary['app'] IS NULL
        ORDER BY committed_at""")


def add_lab_events(rows):
    """Append change events for lab orders. rows: (order_id, region, status, amount, op) tuples."""
    top = spark.sql(f"SELECT max(event_id) FROM {SOURCE_TABLE} WHERE event_id >= {LAB_EVENT_ID_BASE}").first()[0]
    first = (top or LAB_EVENT_ID_BASE) + 1
    df = (spark.createDataFrame(
              [(first + i, o, r, s, Decimal(str(a)), op) for i, (o, r, s, a, op) in enumerate(rows)],
              "event_id BIGINT, order_id BIGINT, region STRING, status STRING, amount DECIMAL(12,2), op STRING")
          .withColumn("event_ts", F.current_timestamp()))
    # Match the table's current columns, so this keeps working after a column is added.
    df = df.select([F.col(f.name) if f.name in df.columns else F.lit(None).cast(f.dataType).alias(f.name)
                    for f in spark.table(SOURCE_TABLE).schema.fields])
    committed(lambda: df.writeTo(SOURCE_TABLE).append())


def probe(label, options=None, since_ms=None, seconds=30, count_where=None, trigger=None):
    """Stream the source table for a while, then report what the stream delivered or why it failed.

    since_ms sets stream-from-timestamp, so the probe starts at the change instead of re-reading the
    whole table. count_where counts the delivered rows matching a SQL condition, to tell whether
    the changed rows came through.
    """
    seen = {"batches": 0, "rows": 0, "matching": 0}
    opts = dict(options or {})
    if since_ms is not None:
        opts["stream-from-timestamp"] = since_ms
    reader = spark.readStream.format("iceberg")
    for key, value in opts.items():
        reader = reader.option(key, str(value).lower() if isinstance(value, bool) else str(value))

    def collect(batch_df, batch_id):
        seen["batches"] += 1
        seen["rows"] += batch_df.count()
        if count_where:
            seen["matching"] += batch_df.where(count_where).count()

    writer = (reader.load(SOURCE_TABLE).writeStream.foreachBatch(collect)
              .option("checkpointLocation", f"{PROBE_DIR}/{label}-{uuid.uuid4().hex[:8]}"))
    writer = writer.trigger(**trigger) if trigger else writer.trigger(processingTime="5 seconds")
    query = writer.start()
    try:
        query.awaitTermination(seconds)
    except StreamingQueryException:
        pass                      # reported below
    finally:
        stream_control.wait_for_idle(query, timeout_seconds=60)   # do not cancel a running batch
        query.stop()
    error = query.exception()
    outcome = f"FAILED: {first_line(error)}" if error else "ok"
    matching = f", {seen['matching']} matching [{count_where}]" if count_where else ""
    print(f"{label:<32} {seen['rows']:>6} rows in {seen['batches']:>3} batches{matching} -> {outcome}")


def adw_lab_orders(where="order_id >= 900000 AND order_id <> 900999"):
    """The lab orders as ADW's ORDERS table has them; is_deleted = 'Y' marks a deleted order."""
    rows = sink.fetch(f"SELECT order_id, status, amount, last_event_id, is_deleted FROM ORDERS WHERE {where} ORDER BY 1")
    for r in rows:
        print("   ORDERS:", r)
    if not rows:
        print("   ORDERS: no rows")


MARKER_ORDER = 900999                         # a lab order that only carries marker events


def wait_for_consumer(timeout_seconds=300):
    """Wait until the consumer of notebook 2 has applied everything appended so far.

    Appends a marker event and waits until ORDERS reflects it. The stream reads appends in
    commit order, so once the marker is in ADW, every earlier append is too. Returns False if
    that does not happen in time -- for example because notebook 2 is not running.
    """
    add_lab_events([(MARKER_ORDER, "lab", "MARKER", 0, "U")])
    marker = spark.sql(f"SELECT max(event_id) FROM {SOURCE_TABLE} WHERE order_id = {MARKER_ORDER}").first()[0]
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        rows = sink.fetch("SELECT last_event_id FROM ORDERS WHERE order_id = :o", o=MARKER_ORDER)
        if rows and rows[0][0] >= marker:
            return True
        time.sleep(5)
    print("The consumer has not caught up -- is notebook 2 running?")
    return False


def resync_snapshot(snapshot_id):
    """Apply a skipped snapshot's changes to ADW.

    Iceberg's changelog view lists the rows the snapshot removed and added; their order_ids are
    the affected orders. For those orders the newest surviving event in the current table is the
    truth, and AdwSink.resync_orders writes it to ADW. Re-running it for the same snapshot does nothing.

    It first waits until the consumer has applied every append so far. Otherwise the consumer
    could still deliver older appends of the affected orders after the resync -- for example
    re-insert orders the snapshot deleted.
    """
    label = f"snapshot-{snapshot_id}"
    if sink.resync_applied(label):
        print(f"resync {label!r} was already applied, skipped")
        return
    if not wait_for_consumer():
        raise RuntimeError("the consumer has not caught up yet; resync later")
    parent = spark.sql(f"SELECT parent_id FROM {SOURCE_TABLE}.snapshots WHERE snapshot_id = {snapshot_id}").first()[0]
    if not spark.sql(f"SELECT 1 FROM {SOURCE_TABLE}.snapshots WHERE snapshot_id = {parent}").first():
        raise RuntimeError(f"snapshot {snapshot_id} or its parent has expired: its changelog is gone")
    view = f"changes_{snapshot_id}"
    spark.sql(f"""CALL {SYSTEM}.create_changelog_view(
                     table => '{SOURCE_TABLE}',
                     options => map('start-snapshot-id', '{parent}', 'end-snapshot-id', '{snapshot_id}'),
                     changelog_view => '{view}')""")
    changes = spark.table(view)
    print("changelog:", {r[0]: r[1] for r in changes.groupBy("_change_type").count().collect()})
    changed = changes.groupBy("order_id").agg(F.max("event_id").alias("changed_max"))
    now = spark.sql(f"SELECT * FROM {SOURCE_TABLE} VERSION AS OF {current_snapshot_id()}")
    newest_first = Window.partitionBy("order_id").orderBy(F.col("event_id").desc())
    truth = (now.join(changed.select("order_id"), "order_id")
             .withColumn("rn", F.row_number().over(newest_first)).where("rn = 1").drop("rn"))
    # Per order, the highest event id this resync looked at: an ADW row above it was updated by
    # an event that arrived later, and the resync leaves it alone.
    watermarks = (changed.join(truth.select("order_id", F.col("event_id").alias("newest")), "order_id", "left")
                  .select("order_id", F.greatest("changed_max", F.coalesce("newest", "changed_max")).alias("watermark_id")))
    sink.resync_orders(truth, watermarks, label=label)


# Records the scratch tables this run creates; the last cell drops those, and only those.
SCRATCH = scratch_tables.ScratchTables(spark, CATALOG, SCHEMA, WAREHOUSE)


def scratch_table(name):
    """A small Iceberg copy of the lab events, for changes that would break notebooks 1 and 2."""
    return SCRATCH.create(name, f"""SELECT event_id, order_id, region, status, amount, op, event_ts
                                    FROM {SOURCE_TABLE} WHERE region LIKE 'lab%'""")


def new_checkpoint(name):
    """A fresh checkpoint folder for a watch() stream."""
    return f"{PROBE_DIR}/{name}-{uuid.uuid4().hex[:8]}"


def watch(table, steps=(), options=None, checkpoint=None, seconds=25):
    """Stream `table` while running `steps` (functions that change it), then print every micro-batch.

    Give two watches the same `checkpoint` to restart the first one's stream where it stopped.
    """
    batches = []
    checkpoint = checkpoint or new_checkpoint("watch")
    reader = spark.readStream.format("iceberg")
    for key, value in (options or {}).items():
        reader = reader.option(key, str(value))

    def collect(batch_df, batch_id):
        first = batch_df.drop("event_ts").limit(1).collect()
        batches.append(f"batch {batch_id}: {batch_df.count()} rows, first row {first[0].asDict() if first else None}")

    def wait(timeout):
        try:
            query.awaitTermination(timeout)
        except StreamingQueryException:
            pass                            # reported below; the remaining steps still run

    query = (reader.load(table).writeStream.foreachBatch(collect)
             .option("checkpointLocation", checkpoint).trigger(processingTime="3 seconds").start())
    try:
        for step in steps:
            wait(10)                        # let the stream catch up before the next change
            step()
        wait(seconds)
    finally:
        stream_control.wait_for_idle(query, timeout_seconds=60)   # do not cancel a running batch
        query.stop()
    for line in batches:
        print("  ", line)
    error = query.exception()
    print("   stream:", f"FAILED: {first_line(error)}" if error else "ok")


def try_sql(statement):
    """Run a statement and print why it failed, if it did."""
    try:
        spark.sql(statement)
        print("ok:", statement.split()[0:4])
    except Exception as e:
        lines = [l for l in str(e).splitlines() if "Exception" in l or "message" in l]
        print("FAILED:", (lines[0] if lines else str(e).splitlines()[0]).strip()[:300])


START_MS = now_ms()
print("consumer is at:", stream_control.committed_source_offset(CONSUMER_CHECKPOINT))

## 0. Seed the lab orders

Twenty lab orders in region `lab`, five in `lab_purge` and five in `lab_overwrite`. This is a plain append, so the consumer picks it up like any producer commit.

In [ ]:
add_lab_events([(900000 + i, "lab", "NEW", 100 + i, "U") for i in range(1, 21)]
               + [(900100 + i, "lab_purge", "NEW", 200 + i, "U") for i in range(1, 6)]
               + [(900200 + i, "lab_overwrite", "NEW", 300 + i, "U") for i in range(1, 6)])
changes_since(START_MS).show(truncate=False)
wait_for_consumer()
print(sink.fetch("SELECT COUNT(*) FROM ORDERS WHERE order_id >= 900000 AND order_id <> :m AND is_deleted = 'N'",
                 m=MARKER_ORDER)[0][0], "lab orders in ADW")

**Observed:** one `append` snapshot of 30 rows; 30 lab orders in `ORDERS` once the consumer has caught up. `wait_for_consumer` appends a marker event for order 900999 and waits until ADW has it: the stream applies appends in commit order, so everything before the marker is in ADW too.

## 1. Appends

The producer's commits are `append` snapshots, the only kind an Iceberg stream reads. ADW sees new commits through the external table **without any refresh**: run the next cell twice and the counts grow.

In [ ]:
probe("appends, default options", since_ms=now_ms() - 30_000)
print("ADW external table:", sink.fetch(f"SELECT COUNT(*), MAX(event_id) FROM {EXTERNAL_TABLE} WHERE event_id < :b", b=LAB_EVENT_ID_BASE)[0])
print("consumer batches:  ", sink.fetch("SELECT COUNT(*), MAX(max_event_id) FROM STREAM_BATCH_LOG WHERE query_name = :q AND max_event_id < :b",
                                        q=CONSUMER_QUERY, b=LAB_EVENT_ID_BASE)[0])

**Observed:** the probe delivers every appended row; both counts in ADW grow from one run of the cell to the next.

## 2. Row-level updates

A copy-on-write `UPDATE` rewrites the data files holding the matching rows, so it commits an **`overwrite`** snapshot. A merge-on-read update -- not used here, see the README -- commits an `overwrite` snapshot too, with delete files.

In [ ]:
t0 = now_ms()
write_sql(f"UPDATE {SOURCE_TABLE} SET status = 'CORRECTED' WHERE order_id IN (900001, 900002, 900003)")
changes_since(t0).show(truncate=False)
probe("update, default options", since_ms=t0, count_where="status = 'CORRECTED'")
probe("update, skip-overwrite", {"streaming-skip-overwrite-snapshots": True}, since_ms=t0, count_where="status = 'CORRECTED'")
probe("update, skip-delete only", {"streaming-skip-delete-snapshots": True}, since_ms=t0, count_where="status = 'CORRECTED'")
print("ADW external table, CORRECTED rows:", sink.fetch(f"SELECT COUNT(*) FROM {EXTERNAL_TABLE} WHERE status = 'CORRECTED'")[0][0])
adw_lab_orders("order_id BETWEEN 900001 AND 900003")

**Observed:**

| | Result |
|---|---|
| snapshot | `overwrite`: it rewrote the whole `lab` data file |
| default options | **fails**: "Cannot process overwrite snapshot: ..., to ignore overwrites, set streaming-skip-overwrite-snapshots=true" |
| `streaming-skip-overwrite-snapshots` | keeps running; **0** corrected rows delivered |
| `streaming-skip-delete-snapshots` alone | fails: it does not cover overwrites |
| ADW external table | shows the 3 corrected rows at once |
| ADW `ORDERS` | still `NEW`: the consumer skipped the snapshot |

No read option delivers row-level changes: an Iceberg stream reads only appended data. Two ways around it follow.

**Alternative A -- write changes as new events.** Instead of updating rows in place, append an event that carries the new state. It is an append, so every stream reads it, and the consumer's MERGE applies it because its `event_id` is newer. This is the model this sample's source table follows.

In [ ]:
add_lab_events([(900004, "lab", "CORRECTED", 104, "U")])
wait_for_consumer()
adw_lab_orders("order_id = 900004")

**Observed:** order 900004 is `CORRECTED` in `ORDERS` after the consumer's next micro-batch.

**Alternative B -- catch up from the changelog.** When the update has already happened, Iceberg's `create_changelog_view` procedure lists what a snapshot changed -- `DELETE`/`INSERT` pairs, or `UPDATE_BEFORE`/`UPDATE_AFTER` -- and `resync_snapshot` rewrites the affected orders in ADW from the current table. It first waits until the consumer has read every append committed before the snapshot: if the consumer still had older appends of those orders to deliver, they would land after the resync and undo it. The view does not support delete files: one more reason for copy-on-write.

In [ ]:
update_snapshot = changes_since(t0).where("operation = 'overwrite'").first().snapshot_id
resync_snapshot(update_snapshot)
adw_lab_orders("order_id BETWEEN 900001 AND 900003")

**Observed:** the changelog has a `DELETE` and an `INSERT` row for each of the 3 changed events -- the other rows of the rewritten file are recognised as unchanged and left out -- and orders 900001-900003 become `CORRECTED` in ADW.

## 3. Deletes

Which snapshot a `DELETE` commits depends on whether it can drop whole files:

- a row-level copy-on-write delete rewrites files: an **`overwrite`** snapshot;
- a delete that matches whole files, such as a whole partition, only removes them: a **`delete`** snapshot. A merge-on-read delete is a `delete` snapshot too.

In [ ]:
t0 = now_ms()
write_sql(f"DELETE FROM {SOURCE_TABLE} WHERE order_id = 900005")           # row-level
changes_since(t0).show(truncate=False)
probe("row delete, default", since_ms=t0)
probe("row delete, skip-delete", {"streaming-skip-delete-snapshots": True}, since_ms=t0)
probe("row delete, skip-overwrite", {"streaming-skip-overwrite-snapshots": True}, since_ms=t0)

t1 = now_ms()
write_sql(f"DELETE FROM {SOURCE_TABLE} WHERE region = 'lab_purge'")        # a whole partition
# A delete snapshot adds no files: a stream starting at it sees nothing new until a later snapshot
# exists. Append a marker event so the probes must read past the delete.
add_lab_events([(MARKER_ORDER, "lab", "MARKER", 0, "U")])
changes_since(t1).show(truncate=False)
probe("partition delete, default", since_ms=t1)
probe("partition delete, skip-overwrite", {"streaming-skip-overwrite-snapshots": True}, since_ms=t1)
probe("partition delete, skip-delete", {"streaming-skip-delete-snapshots": True}, since_ms=t1);

**Observed:**

| | row-level `DELETE` | partition `DELETE` |
|---|---|---|
| snapshot | `overwrite` | `delete` |
| default options | fails: "Cannot process overwrite snapshot" | fails: "Cannot process delete snapshot" |
| `streaming-skip-delete-snapshots` | **still fails** | ok |
| `streaming-skip-overwrite-snapshots` | ok | fails |

So a pipeline that must survive deletes needs both options, and either way the stream delivers nothing about the deleted rows. A stream that starts exactly at a `delete` snapshot does not fail until a later snapshot arrives, since the delete adds no files -- hence the marker append above. The same two ways around it apply: a delete *event* -- the consumer's MERGE marks the order deleted for `op = 'D'` -- or a catch-up from the changelog.

In [ ]:
add_lab_events([(900006, "lab", "CANCELLED", 106, "D")])          # alternative A: a delete event
for snapshot in changes_since(t0).where("operation IN ('overwrite', 'delete')").collect():
    resync_snapshot(snapshot.snapshot_id)                            # alternative B: catch up
wait_for_consumer()
adw_lab_orders("order_id IN (900005, 900006) OR order_id BETWEEN 900101 AND 900105")

**Observed:** 900005, 900006 and the five `lab_purge` orders are all marked `is_deleted = 'Y'` in `ORDERS`.

## 4. Overwrites

`INSERT OVERWRITE` -- static or dynamic, and `DataFrameWriterV2.overwritePartitions()` -- commits an `overwrite` snapshot that removes the old rows **and adds new ones**. Skipping the snapshot skips the new rows as well: silent data loss downstream.

In [ ]:
t0 = now_ms()
write_sql(f"""INSERT OVERWRITE {SOURCE_TABLE} PARTITION (region = 'lab_overwrite')
              SELECT {LAB_EVENT_ID_BASE} + 500000 + id, 900300 + id, 'REPLACED', 50.00, 'U', current_timestamp()
              FROM range(1, 4)""")
changes_since(t0).show(truncate=False)
probe("overwrite, skip-overwrite", {"streaming-skip-overwrite-snapshots": True}, since_ms=t0, count_where="status = 'REPLACED'")
wait_for_consumer()
adw_lab_orders("order_id BETWEEN 900201 AND 900205 OR order_id BETWEEN 900301 AND 900303")

**Observed:** an `overwrite` snapshot that deleted 5 rows and added 3. The probe keeps running and delivers **0** of the 3 `REPLACED` rows; `ORDERS` still has the 5 old orders as live rows and none of the 3 new ones.

The catch-up from the changelog fixes both sides: it deletes the removed orders and inserts the new ones.

In [ ]:
resync_snapshot(changes_since(t0).where("operation = 'overwrite'").first().snapshot_id)
adw_lab_orders("order_id BETWEEN 900201 AND 900205 OR order_id BETWEEN 900301 AND 900303")

**Observed:** orders 900201-900205 are marked deleted in `ORDERS`, 900301-900303 are there as `REPLACED`.

## 5. MERGE

A `MERGE` that updates rows rewrites files: an `overwrite` snapshot, like an update. A `MERGE` with **only** a `WHEN NOT MATCHED THEN INSERT` clause is planned as a plain append, so streams read it -- a handy way to append events idempotently, for example when an upstream system may send the same event twice.

In [ ]:
new_events = spark.sql(f"""
    SELECT {LAB_EVENT_ID_BASE} + 600000 + id AS event_id, 900010 + id AS order_id, 'lab' AS region,
           'MERGED' AS status, CAST(10 AS DECIMAL(12,2)) AS amount, 'U' AS op, current_timestamp() AS event_ts
    FROM range(0, 3)""")
new_events.createOrReplaceTempView("incoming_events")

t0 = now_ms()
write_sql(f"""MERGE INTO {SOURCE_TABLE} t USING incoming_events s ON t.event_id = s.event_id
              WHEN NOT MATCHED THEN INSERT *""")                                     # insert-only
changes_since(t0).show(truncate=False)
probe("insert-only MERGE, default", since_ms=t0, seconds=20, count_where="status = 'MERGED'")

t1 = now_ms()
write_sql(f"""MERGE INTO {SOURCE_TABLE} t USING incoming_events s ON t.event_id = s.event_id
              WHEN MATCHED THEN UPDATE SET status = 'MERGED_AGAIN'
              WHEN NOT MATCHED THEN INSERT *""")                                     # upsert
changes_since(t1).show(truncate=False)
probe("upsert MERGE, skip-overwrite", {"streaming-skip-overwrite-snapshots": True}, since_ms=t1, count_where="status = 'MERGED_AGAIN'")
resync_snapshot(changes_since(t1).where("operation = 'overwrite'").first().snapshot_id)
adw_lab_orders("order_id BETWEEN 900010 AND 900012")

**Observed:** the insert-only `MERGE` commits an `append` snapshot and a stream with default options delivers its 3 rows; the upsert `MERGE` commits an `overwrite` snapshot, delivers nothing under `skip-overwrite`, and needs the catch-up. After it, orders 900010-900012 are `MERGED_AGAIN` in ADW.

## 6. Schema evolution

### 6a. Compatible: add a column -- on the live table

In [ ]:
write_sql(f"ALTER TABLE {SOURCE_TABLE} ADD COLUMN channel STRING")
t0 = now_ms()
add_lab_events([(900007, "lab", "PAID", 107, "U")])
probe("new probe after ADD COLUMN", since_ms=t0, seconds=20, count_where="channel IS NULL")
print("ADW external table columns:", [r[0] for r in sink.fetch(
    "SELECT column_name FROM user_tab_columns WHERE table_name = :t ORDER BY column_id", t=EXTERNAL_TABLE)])
sink.create_iceberg_external_table(EXTERNAL_TABLE, ADW_OS_CREDENTIAL, TABLE_LOCATION, OCI_REGION, replace=True)
print("after replace:             ", [r[0] for r in sink.fetch(
    "SELECT column_name FROM user_tab_columns WHERE table_name = :t ORDER BY column_id", t=EXTERNAL_TABLE)])

**Observed:**

- The **producer keeps running**; its rows have `channel = NULL`.
- A **running** stream keeps the schema it started with: rows committed after the change arrive without `channel`, even rows that have a value in it. It sees the column after a restart from its checkpoint. A new probe sees it at once.
- The consumer selects the columns it needs, so it is unaffected; to carry `channel` to ADW, add the column to `ORDERS` and to the MERGE, then restart the consumer.
- The ADW external table keeps working with its old column list; `replace=True` adds `CHANNEL`.

### 6b. Rename and drop -- on a scratch copy

On the live table these would stop the producer: we saw it fail with "Field status not found in source schema" after a rename, and likewise after a drop, also after a restart, until its code changes. So watch a stream over a scratch copy instead.

In [ ]:
lab = scratch_table("schema_lab")
checkpoint = new_checkpoint("schema_lab")
watch(lab, checkpoint=checkpoint, steps=[
    lambda: spark.sql(f"ALTER TABLE {lab} RENAME COLUMN status TO order_status"),
    lambda: spark.sql(f"""INSERT INTO {lab} (event_id, order_id, region, order_status, amount, op, event_ts)
                          SELECT event_id + 1, order_id, region, 'RENAMED', amount, op, current_timestamp() FROM {lab} LIMIT 2"""),
    lambda: spark.sql(f"ALTER TABLE {lab} DROP COLUMN amount"),
    lambda: spark.sql(f"""INSERT INTO {lab} (event_id, order_id, region, order_status, op, event_ts)
                          SELECT event_id + 2, order_id, region, 'DROPPED', op, current_timestamp() FROM {lab} LIMIT 2"""),
])
print("restarted from the same checkpoint:")
watch(lab, [lambda: spark.sql(f"""INSERT INTO {lab} (event_id, order_id, region, order_status, op, event_ts)
                                   SELECT event_id + 3, order_id, region, 'AFTER', op, current_timestamp() FROM {lab} LIMIT 2""")],
      checkpoint=checkpoint)

**Observed:** while running, the stream keeps the columns it started with: after the rename the new value `RENAMED` still arrives -- Iceberg tracks columns by ID -- under the old name `status`; after the drop, `amount` arrives as `None`. Restarted from the same checkpoint, it delivers the new schema: `order_status`, no `amount`. Partition evolution -- `ALTER TABLE ... ADD PARTITION FIELD` -- was transparent to the stream.

### 6c. Incompatible: change a column type -- on a scratch copy

AIDP rejects every column type change on an Iceberg table, even widenings that Iceberg allows, such as `INT` to `BIGINT`, `FLOAT` to `DOUBLE` or a wider `DECIMAL`.

In [ ]:
try_sql(f"ALTER TABLE {lab} ALTER COLUMN order_id TYPE DECIMAL(20,0)")
try_sql(f"ALTER TABLE {lab} ALTER COLUMN order_id TYPE STRING")

**Observed:** both fail with "405 ... Changing Column Data Type operation is not supported for this data format".

The workarounds:

1. **Add a new column** with the new type, write both for a while, move readers to the new one, then drop the old one -- every step is a compatible change.
2. **Rebuild the table** with `CREATE OR REPLACE TABLE ... AS SELECT`, which AIDP allows. The rebuild commits a snapshot with no parent, and a running stream fails on it. Readers then start over on a new checkpoint from just after the rebuild; for the consumer of notebook 2, follow "Starting over on a new checkpoint" in the README: reload ADW from the rebuilt table with `AdwSink.reload`, then start the stream with `stream-from-timestamp` after that snapshot.

In [ ]:
rebuilt = {}

def rebuild():
    spark.sql(f"""CREATE OR REPLACE TABLE {lab} USING iceberg AS
                  SELECT event_id, CAST(order_id AS STRING) AS order_id, region, order_status, op, event_ts FROM {lab}""")
    rebuilt["ms"] = spark.sql(f"SELECT max(unix_millis(committed_at)) FROM {lab}.snapshots").first()[0]

watch(lab, [rebuild, lambda: spark.sql(f"INSERT INTO {lab} SELECT event_id + 9, order_id, region, 'AFTER_REBUILD', op, event_ts FROM {lab} LIMIT 2")])
print("new checkpoint, starting just after the rebuild commit:")
watch(lab, [lambda: spark.sql(f"INSERT INTO {lab} SELECT event_id + 10, order_id, region, 'NEW_STREAM', op, event_ts FROM {lab} LIMIT 2")],
      options={"stream-from-timestamp": rebuilt["ms"] + 1})

**Observed:** the running stream fails with `Cannot invoke "java.lang.Long.longValue()" because the return value of "org.apache.iceberg.Snapshot.parentId()" is null` as soon as the rebuild commits. The new stream, starting just after the rebuild commit, delivers only the rows appended afterwards -- `AFTER_REBUILD` and `NEW_STREAM` -- with `order_id` now a string. A `rollback_to_snapshot` breaks a running stream in the same way, and rows the stream already delivered from the rolled-back snapshots stay delivered.

## 7. Table restructuring

### 7a. Compaction and manifest rewrites -- on the live table

Streaming leaves many small files: here one per partition per commit. `rewrite_data_files` and `rewrite_manifests` commit **`replace`** snapshots, which an Iceberg stream skips: they change the files, not the data.

In [ ]:
t0 = now_ms()
files_before = spark.sql(f"SELECT count(*) FROM {SOURCE_TABLE}.files").first()[0]
write_sql(f"CALL {SYSTEM}.rewrite_data_files(table => '{SOURCE_TABLE}', options => map('min-input-files', '2'))").show()
write_sql(f"CALL {SYSTEM}.rewrite_manifests(table => '{SOURCE_TABLE}')").show()
print("data files:", files_before, "->", spark.sql(f"SELECT count(*) FROM {SOURCE_TABLE}.files").first()[0])
changes_since(t0).show(truncate=False)
probe("after compaction, default", since_ms=t0)

**Observed:** two `replace` snapshots; the probe runs with default options and delivers only the producer's new appends -- no duplicates of the rewritten rows. The consumer and the ADW external table carry on. Section 9 checks that no order was duplicated or lost.

### 7b. Expiring snapshots without breaking the consumer

`expire_snapshots` deletes snapshots and the files only they reference. A query keeps a position in its checkpoint -- a snapshot id and a file position -- and needs that snapshot and every later one. Expire a snapshot that a stopped or lagging query still needs, and that query **fails on restart** with "Cannot load current offset at snapshot ..., the snapshot was expired or removed".

So expire only snapshots older than the oldest position of every query that reads the table, and only once every skipped snapshot has been caught up: `resync_snapshot` needs the snapshot and its parent. `stream_control.committed_source_offset` reads a query's position from its checkpoint:

In [ ]:
position = stream_control.committed_source_offset(CONSUMER_CHECKPOINT)
print("consumer position:", position)
needed_since = spark.sql(f"SELECT committed_at FROM {SOURCE_TABLE}.snapshots WHERE snapshot_id = {position['snapshot_id']}").first()[0]
print("expiring snapshots older than", needed_since)
write_sql(f"""CALL {SYSTEM}.expire_snapshots(table => '{SOURCE_TABLE}',
                                            older_than => TIMESTAMP '{needed_since}', retain_last => 20)""").show()

With more than one reader, take the oldest of their positions, and stop readers while you expire so that none of them falls behind the cut-off in the meantime.

On a scratch copy, the failure, and a second trap: **a new query started after compaction and expiry reads nothing**. A new query starts at the table's oldest snapshot and reads the files that snapshot *added*. After compaction plus expiry, the oldest snapshot is the compaction's `replace` snapshot, which a stream skips. The way around it: load the table once in batch, then stream from just after that snapshot.

In [ ]:
lab = scratch_table("expiry_lab")
checkpoint = new_checkpoint("expiry_lab")
watch(lab, checkpoint=checkpoint)                          # a query that reads the table, then stops
spark.sql(f"INSERT INTO {lab} SELECT event_id + 100, order_id, region, status, amount, op, event_ts FROM {lab}")
spark.sql(f"CALL {SYSTEM}.rewrite_data_files(table => '{lab}', options => map('min-input-files', '2'))")
expire_until = spark.sql("SELECT current_timestamp()").first()[0]
spark.sql(f"CALL {SYSTEM}.expire_snapshots(table => '{lab}', older_than => TIMESTAMP '{expire_until}', retain_last => 1)")
print("snapshots left:", [tuple(r) for r in spark.sql(f"SELECT snapshot_id, operation FROM {lab}.snapshots").collect()])
print("restart the old query:")
watch(lab, checkpoint=checkpoint)
print("a new query:")
watch(lab)

print("bootstrap: batch load, then stream from just after the loaded snapshot")
loaded = spark.sql(f"SELECT snapshot_id, unix_millis(committed_at) AS ms FROM {lab}.snapshots ORDER BY committed_at DESC LIMIT 1").first()
print("   batch load:", spark.sql(f"SELECT * FROM {lab} VERSION AS OF {loaded.snapshot_id}").count(), "rows")
watch(lab, [lambda: spark.sql(f"INSERT INTO {lab} SELECT event_id + 200, order_id, region, status, amount, op, event_ts FROM {lab} LIMIT 3")],
      options={"stream-from-timestamp": loaded.ms + 1})

**Observed:** the restarted query fails with "Cannot load current offset at snapshot ..., the snapshot was expired or removed"; the new query runs but delivers **0 rows** although the table holds every row it ever had -- 60 in one run, 78 in another; the bootstrap loads them all in batch, and the stream then delivers the 3 new ones.

### 7c. Delete files and ADW

This sample's table is copy-on-write. If a table gets merge-on-read deletes, the ADW external table fails with ORA-29913, "Delete files not supported". In our test, `rewrite_data_files` with `rewrite-all` alone left the delete file referenced; adding `remove-dangling-deletes` removed it and ADW could read the table again:

```sql
CALL default.system.rewrite_data_files(table => '<table>', options => map('rewrite-all', 'true', 'remove-dangling-deletes', 'true'))
```

## 8. Read options lab

Probes over the last three minutes of the producer's commits. Each commit adds one data file of about 300 rows per region.

In [ ]:
since = now_ms() - 180_000
probe("no limits", since_ms=since)
probe("max-files-per-micro-batch 1", {"streaming-max-files-per-micro-batch": 1}, since_ms=since, seconds=75)
probe("max-rows-per-micro-batch 700", {"streaming-max-rows-per-micro-batch": 700}, since_ms=since, seconds=60)
probe("max-rows-per-micro-batch 100", {"streaming-max-rows-per-micro-batch": 100}, since_ms=since)
probe("availableNow, max-files 1", {"streaming-max-files-per-micro-batch": 1}, since_ms=since, trigger={"availableNow": True})

**Observed** on Iceberg 1.8.x; newer Iceberg releases behave differently, see the README:

| Probe | Result |
|---|---|
| no limits | 3,600 rows in one batch: all three commits at once |
| `max-files-per-micro-batch 1` | 16 batches of 300 rows: one data file per batch |
| `max-rows-per-micro-batch 700` | 10 batches of 600 rows: whole files only, two 300-row files fit under the cap |
| `max-rows-per-micro-batch 100` | **0 rows and no error**: no file fits under the cap, so the stream never advances |
| `availableNow` with `max-files 1` | **one batch of 300 rows, then the query ends**. Run again from the same checkpoint, it made no progress; a `processingTime` trigger on that checkpoint picked up the rest |

## 9. Stop the streams and reconcile

Stop the producer and wait until it is gone, let the consumer catch up, apply any skipped snapshot this notebook has not caught up yet (resyncs are idempotent and need the consumer running), stop the consumer, and compare ADW with the source.

In [ ]:
def wait_until_stopped(query_name, timeout_seconds=300):
    """Wait until no streaming query called `query_name` runs on the cluster."""
    deadline = time.time() + timeout_seconds
    while any(q["name"] == query_name for q in stream_control.list_cluster_streams(spark)):
        if time.time() > deadline:
            raise RuntimeError(f"{query_name} is still running")
        time.sleep(5)


stream_control.request_stop(CONTROL_DIR, "producer")
wait_until_stopped("iceberg_producer")            # it finishes its current micro-batch first
if not wait_for_consumer():
    raise RuntimeError("The consumer has not caught up; is notebook 2 still running?")

# Catch up skipped snapshots while the consumer still runs: each resync waits for it first.
for snapshot in changes_since(START_MS).where("operation IN ('overwrite', 'delete')").collect():
    try:
        resync_snapshot(snapshot.snapshot_id)             # does nothing for snapshots already caught up
    except RuntimeError as e:
        print("not caught up:", e)

stream_control.request_stop(CONTROL_DIR, "consumer")
wait_until_stopped(CONSUMER_QUERY)
print("both streams stopped")
print(sink.compare_with_source(EXTERNAL_TABLE))

**Observed:** `missing_in_adw`, `extra_in_adw` and `different` are all 0: every order in ADW matches the newest surviving event in the Iceberg table, across appends, updates, deletes, overwrites, merges, a new column, compaction and snapshot expiry.

## Clean up the scratch tables -- optional

Drops the scratch tables **this run** created, and deletes their folders: `ScratchTables` records each name and location as it creates them. Other tables are never touched, even ones whose names start with `scratch_`. If the kernel restarted since sections 6 and 7 ran, the record is gone and nothing is dropped; the earlier tables' names were printed when they were created.

In [ ]:
CLEAN_UP = False   # set to True to drop the scratch tables this run created, and their folders

if CLEAN_UP:
    SCRATCH.drop_all()
else:
    print("would drop:", [table for table, _ in SCRATCH.created])